# Лабораторная работа № 3
## Распределённый информационный поиск: шардирование индекса, брокер и слияние результатов

**Модуль 3.** Распределённый информационный поиск · **Дисциплина:** «Информационный поиск и базы данных» · **Направление:** 44.03.05 Педагогическое образование, профили «Информатика» и «Английский язык» · **Максимум:** 20 баллов (обязательная часть 16 + усложнение 4)

**Среда:** Google Colab, Python 3.10+ · **Данные:** NFCorpus (BEIR) — 3 633 англоязычных научных документа о питании и здоровье, 323 тестовых запроса с оценками релевантности 0 / 1 / 2. **Усложнение:** MSLR (Microsoft Learning to Rank) из семинара.

**Цель.** Понять, как работает поиск, когда индекс не помещается на один сервер: разбить коллекцию на шарды, построить брокер, который рассылает запрос и сливает ответы, измерить, как на качество влияют локальная статистика, способ слияния, выбор шардов и отказ сервера. **Усложнение** — обучить модель ранжирования на брокере по материалам семинара.

> Машинное обучение в обязательной части **не нужно**: поиск строится на формуле BM25, а качество измеряется метрикой NDCG. Модель ранжирования появляется только в усложнении, и её код дан полностью.

### Как работать с ноутбуком

| Что видите в ячейке | Что сделать |
|---|---|
| `V = None` | впишите номер варианта (целое число 1–20) |
| `...` и справа `# ← впишите: …` | сотрите три точки и впишите код из подсказки (можно скопировать) |
| `Не заполнено (…)` после запуска | в ячейке осталось `...`; в сообщении указано, какая строка |
| **«Ячейка готова — просто запустите»** | ничего не меняйте, нажмите **Shift + Enter** |
| **«Как читать результат»** | прочитайте перед выводом |
| **«Что получено»** | дважды щёлкните и допишите предложения своими числами — это основная часть отчёта |

Ошибка `NameError: name '…' is not defined` — не запущена одна из **предыдущих** ячеек: **Runtime → Run before**. Перед сдачей — **Runtime → Restart and run all**.

| Где дописать код | Что вписать |
|---|---|
| Шаг 0 | ФИО, группа, номер варианта, гипотеза |
| Шаг 2, TODO 1–2 | список вхождений (posting list) и формула IDF |
| Шаг 4, TODO 3 | разбиение коллекции на шарды |
| Шаг 5, TODO 4 | функция слияния RRF |
| Шаг 6, TODO 5 | выбранное число опрашиваемых шардов |
| Усложнение, TODO Б1–Б2 | валидационный набор и функция потерь модели |

### Критерии оценивания (20 баллов)

| Баллы | Что оценивается | Шаги |
|---:|---|---|
| 3 | инвертированный индекс и BM25 на игрушечном примере; локальная и глобальная статистика | 2 |
| 2 | централизованный эталон и шардирование коллекции | 3–4 |
| 4 | брокер и слияние результатов: сравнение шести методов, объяснение различий | 5 |
| 3 | выбор шардов: компромисс «качество — стоимость — задержка» | 6 |
| 2 | отказ шарда и репликация; анализ выдачи по англоязычному запросу варианта | 7–8 |
| 2 | итоговый вывод, ответы на контрольные вопросы, оформление | итог |
| 4 | **усложнение**: модель ранжирования на брокере (MSLR) | Б1–Б4 |

## Теоретический минимум

### Зачем распределять поиск

Индекс крупной поисковой системы занимает терабайты и обслуживает тысячи запросов в секунду — один сервер не справится. Поэтому коллекцию делят на части — **шарды** (shards) — и размещают на разных серверах. Запрос принимает **брокер** (broker, aggregator): он рассылает запрос всем нужным шардам, получает от каждого его лучшие k документов и **сливает** их в одну выдачу. Схема называется **scatter–gather** («разослать — собрать»).

```text
                  ┌── шард 1: локальный индекс → топ-k ──┐
запрос → брокер ──┼── шард 2: локальный индекс → топ-k ──┼──→ слияние → итоговая выдача
                  └── шард N: локальный индекс → топ-k ──┘
```

| Понятие | Смысл | Пример в реальных системах |
|---|---|---|
| **Разбиение по документам** | каждый шард хранит полный индекс своей части документов | Elasticsearch, OpenSearch, Solr — основной способ |
| **Разбиение по термам** | каждый шард хранит списки вхождений части слов | редко: сложно балансировать нагрузку |
| **Реплика** | копия шарда на другом сервере | отказоустойчивость и рост пропускной способности |
| **Выбор шардов** | брокер опрашивает только «перспективные» шарды | selective search, федеративный поиск |
| **Слияние результатов** | объединение списков от разных шардов | result merging, metasearch |

### Инвертированный индекс и BM25

**Инвертированный индекс** — словарь «терм → список документов, где он встречается (с частотой)». По нему не нужно просматривать все документы: достаточно взять списки для слов запроса.

**BM25** — стандартная формула оценки документа $d$ по запросу $q$:
$$\mathrm{BM25}(q,d)=\sum_{t\in q}\mathrm{IDF}(t)\cdot\frac{tf_{t,d}\,(k_1+1)}{tf_{t,d}+k_1\left(1-b+b\,\frac{|d|}{avgdl}\right)},\qquad \mathrm{IDF}(t)=\ln\!\left(1+\frac{N-df_t+0{,}5}{df_t+0{,}5}\right).$$

$tf_{t,d}$ — сколько раз слово встречается в документе, $df_t$ — в скольких документах встречается слово, $N$ — число документов, $|d|$ — длина документа, $avgdl$ — средняя длина; $k_1=1{,}2$, $b=0{,}75$. Редкие слова получают большой IDF и сильнее влияют на оценку.

**Главная проблема распределённого BM25:** $N$, $df_t$ и $avgdl$ — это **статистика коллекции**. Если каждый шард считает её только по своим документам (**локальная статистика**), одно и то же слово получает в разных шардах разный IDF, и оценки разных шардов становятся несравнимыми. В Elasticsearch режим по умолчанию `query_then_fetch` использует локальную статистику шардов, а режим `dfs_query_then_fetch` сначала собирает **глобальную** статистику со всех шардов.

### Методы слияния результатов

| Метод | Как считается итоговая оценка | Когда работает |
|---|---|---|
| сырые оценки (локальная статистика) | BM25 шарда как есть | шарды похожи по составу |
| сырые оценки (глобальная статистика) | BM25 с общими $N$, $df$, $avgdl$ | всегда, но нужен обмен статистикой |
| min-max | $s'=\dfrac{s-s_{\min}}{s_{\max}-s_{\min}}$ внутри шарда | оценки разных шкал |
| z-оценка | $s'=\dfrac{s-\bar s}{\sigma_s}$ внутри шарда | оценки разных шкал |
| **RRF** (Reciprocal Rank Fusion) | $\mathrm{RRF}(d)=\dfrac{1}{60+\mathrm{rank}(d)}$ — учитывается только место | оценки вообще несравнимы |
| круговой (round robin) | по очереди: первый из каждого шарда, затем вторые… | простейший метапоиск |

Ловушка нормализации: min-max делает **лучший документ любого шарда** равным 1 — даже если в шарде нет ничего релевантного.

### Выбор шардов (selective search)

Если шарды собраны **по темам**, большинство релевантных документов лежит в нескольких из них. Брокер оценивает шарды как «большие документы» (шард = все его тексты) и опрашивает только $s$ лучших. Выигрыш — меньше вычислений и задержки, риск — пропустить релевантный документ. Задержка брокера равна задержке **самого медленного** опрошенного шарда — поэтому чем меньше шардов, тем меньше «хвост» задержек.

### Метрики качества

* $\mathrm{NDCG@k}$ — качество верхних $k$ позиций выдачи с учётом степени релевантности (0, 1, 2) и позиции:
$$\mathrm{DCG@k}=\sum_{i=1}^{k}\frac{2^{rel_i}-1}{\log_2(i+1)},\qquad \mathrm{NDCG@k}=\frac{\mathrm{DCG@k}}{\mathrm{IDCG@k}}\in[0,1],$$
где $\mathrm{IDCG@k}$ — DCG идеального порядка. 1 — идеальная выдача, 0 — в топ-$k$ нет релевантных.
* $\mathrm{Recall@k}$ — доля всех релевантных документов, попавших в топ-$k$.
* **Стоимость** — доля документов коллекции, которые пришлось оценить; **задержка** — время ответа брокера.

---
# Шаг 0. Студент и вариант

**Что вписать:** ФИО и группу в кавычках, номер варианта — **целым числом без кавычек** (например, `V = 7`). Параметры распределённой системы и запрос для анализа подставятся из таблицы вариантов.

### ✍️ Гипотеза (заполните до работы)

> Как, по-вашему, изменится качество поиска, если разбить индекс на шарды вашего варианта? Какой способ слияния окажется лучше и почему? 1–2 предложения.
>
> _[ваша гипотеза]_

In [ ]:
# =========== ЗАПОЛНИТЕ ===========
STUDENT_NAME  = "Иванов Иван Иванович"   # TODO: ← впишите свои ФИО (в кавычках)
STUDENT_GROUP = "ГР-00"                  # TODO: ← впишите группу (в кавычках)
V = None                                 # TODO: ← впишите номер варианта, например V = 7
# =================================

VARIANTS = {
    1: dict(n_shards=3, strategy="random", k_local=10, k=10, s_select=1, failed=0, ltr_loss="YetiRank", query_id="PLAIN-2600"),
    2: dict(n_shards=4, strategy="topic", k_local=20, k=10, s_select=2, failed=1, ltr_loss="LambdaMart", query_id="PLAIN-3432"),
    3: dict(n_shards=5, strategy="length", k_local=10, k=5, s_select=2, failed=2, ltr_loss="QueryRMSE", query_id="PLAIN-2570"),
    4: dict(n_shards=6, strategy="topic", k_local=10, k=10, s_select=2, failed=3, ltr_loss="PairLogit", query_id="PLAIN-175"),
    5: dict(n_shards=8, strategy="random", k_local=20, k=20, s_select=4, failed=4, ltr_loss="RMSE", query_id="PLAIN-2770"),
    6: dict(n_shards=4, strategy="length", k_local=30, k=10, s_select=2, failed=0, ltr_loss="YetiRank", query_id="PLAIN-1262"),
    7: dict(n_shards=5, strategy="topic", k_local=20, k=5, s_select=2, failed=4, ltr_loss="LambdaMart", query_id="PLAIN-259"),
    8: dict(n_shards=6, strategy="random", k_local=10, k=10, s_select=3, failed=5, ltr_loss="QueryRMSE", query_id="PLAIN-207"),
    9: dict(n_shards=3, strategy="topic", k_local=30, k=20, s_select=1, failed=2, ltr_loss="PairLogit", query_id="PLAIN-3281"),
    10: dict(n_shards=8, strategy="length", k_local=10, k=10, s_select=4, failed=7, ltr_loss="RMSE", query_id="PLAIN-2630"),
    11: dict(n_shards=4, strategy="random", k_local=50, k=5, s_select=2, failed=3, ltr_loss="YetiRank", query_id="PLAIN-2930"),
    12: dict(n_shards=6, strategy="topic", k_local=20, k=20, s_select=3, failed=0, ltr_loss="LambdaMart", query_id="PLAIN-112"),
    13: dict(n_shards=5, strategy="random", k_local=30, k=10, s_select=3, failed=1, ltr_loss="QueryRMSE", query_id="PLAIN-2740"),
    14: dict(n_shards=8, strategy="topic", k_local=10, k=5, s_select=3, failed=6, ltr_loss="PairLogit", query_id="PLAIN-2650"),
    15: dict(n_shards=3, strategy="length", k_local=20, k=10, s_select=2, failed=1, ltr_loss="RMSE", query_id="PLAIN-2490"),
    16: dict(n_shards=7, strategy="topic", k_local=20, k=10, s_select=3, failed=2, ltr_loss="YetiRank", query_id="PLAIN-2710"),
    17: dict(n_shards=7, strategy="random", k_local=10, k=20, s_select=3, failed=6, ltr_loss="LambdaMart", query_id="PLAIN-3462"),
    18: dict(n_shards=5, strategy="length", k_local=50, k=20, s_select=3, failed=3, ltr_loss="QueryRMSE", query_id="PLAIN-56"),
    19: dict(n_shards=4, strategy="topic", k_local=10, k=5, s_select=1, failed=2, ltr_loss="PairLogit", query_id="PLAIN-2560"),
    20: dict(n_shards=8, strategy="topic", k_local=20, k=10, s_select=3, failed=2, ltr_loss="YetiRank", query_id="PLAIN-2510"),
}

assert V is not None, "Вариант: впишите номер варианта вместо None, например V = 7"
assert isinstance(V, int) and 1 <= V <= 20, "Вариант: номер — целое число от 1 до 20, без кавычек"
CFG = VARIANTS[V]
N_SHARDS, STRATEGY, K_LOCAL, K = CFG["n_shards"], CFG["strategy"], CFG["k_local"], CFG["k"]
STRAT_RU = {"random": "случайное", "topic": "тематическое (K-Means)", "length": "по длине документа"}
print(f"Студент: {STUDENT_NAME}, группа {STUDENT_GROUP}, вариант {V}")
print(f"  шардов: {N_SHARDS}; разбиение: {STRAT_RU[STRATEGY]}; каждый шард возвращает топ-{K_LOCAL}")
print(f"  метрика: NDCG@{K}; опрашиваемых шардов при выборе: {CFG['s_select']}; отказывает шард № {CFG['failed']}")
print(f"  усложнение: функция потерь {CFG['ltr_loss']}; запрос для анализа: {CFG['query_id']}")

---
# Шаг 1. Загрузка коллекции NFCorpus

**Ячейка готова — просто запустите.** Код скачивает архив по HTTPS (сначала из репозитория курса, при недоступности — с официального сервера BEIR), читает документы, запросы и оценки релевантности. Документ = заголовок + аннотация научной статьи; запрос — заголовок материала сайта NutritionFacts на английском языке.

In [ ]:
import io, json, math, re, time, urllib.request, warnings, zipfile
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 90); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
C_MAIN, C_ALT, C_BAD = "#4c2a85", "#e07b39", "#c0392b"

URLS = [
    "https://raw.githubusercontent.com/BosenkoTM/IIR/main/labs/lab_03/data/nfcorpus.zip",   # репозиторий курса
    "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/nfcorpus.zip",    # официальный сервер BEIR
]

def check_filled(where, values):
    """Проверяет, что вместо «...» вписан код; если нет — называет незаполненную строку."""
    empty = [name for name, v in values.items() if v is ...]
    if empty:
        raise ValueError(f"Не заполнено ({where}): " + ", ".join(empty)
                         + ". Замените «...» на код из подсказки «← впишите:» и запустите ячейку снова.")

def load_nfcorpus(urls):
    errors = []
    for url in urls:
        try:
            with urllib.request.urlopen(url, timeout=60) as r:
                z = zipfile.ZipFile(io.BytesIO(r.read()))
            read = lambda name: z.read("nfcorpus/" + name).decode("utf-8")
            docs = pd.DataFrame([json.loads(l) for l in read("corpus.jsonl").splitlines()])
            queries = {q["_id"]: q["text"] for q in map(json.loads, read("queries.jsonl").splitlines())}
            qrels = pd.read_csv(io.StringIO(read("qrels/test.tsv")), sep="\t")
            qrels.columns = ["qid", "doc_id", "rel"]
            return docs.rename(columns={"_id": "doc_id"})[["doc_id", "title", "text"]], queries, qrels, url
        except Exception as exc:
            errors.append(f"{url} → {type(exc).__name__}: {exc}")
    raise RuntimeError("Не удалось загрузить NFCorpus:\n" + "\n".join(errors))

docs, queries, qrels, USED_URL = load_nfcorpus(URLS)
docs["content"] = docs["title"] + ". " + docs["text"]
test_q = sorted(qrels["qid"].unique())
REL = {q: dict(zip(g["doc_id"], g["rel"])) for q, g in qrels.groupby("qid")}
print("Источник:", USED_URL)
print(f"Документов: {len(docs)}, тестовых запросов: {len(test_q)}, оценок релевантности: {len(qrels)}")
print("Оценки:", qrels["rel"].value_counts().sort_index().to_dict(), "· релевантных на запрос (медиана):", int(qrels.groupby("qid").size().median()))
print("\nПример запроса:", CFG["query_id"], "—", queries[CFG["query_id"]])
docs.head(3)

**Как читать результат.** Каждый документ — научная аннотация, каждый запрос — короткая англоязычная фраза. Оценка 2 — документ очень релевантен, 1 — релевантен, отсутствие оценки — нерелевантен. У одного запроса в среднем несколько десятков релевантных документов: это коллекция для оценки **ранжирования**, а не поиска одного «правильного» ответа.

**Что получено.** Допишите:

> В коллекции ___ документов и ___ тестовых запросов; оценок «2» — ___, оценок «1» — ___. Запрос моего варианта: «___» (перевод: «___»).

---
# Шаг 2. Инвертированный индекс и BM25 на игрушечном примере (TODO 1–2)

Прежде чем работать с тысячами документов, разберём механику на восьми коротких фразах, разложенных по **двум шардам**.

**Задание.**

* **TODO 1** — допишите строку, которая добавляет документ в список вхождений терма (и считает частоту):
  `index[term][doc_id] = index[term].get(doc_id, 0) + 1`
* **TODO 2** — допишите формулу IDF:
  `math.log(1 + (n_docs - df + 0.5) / (df + 0.5))`

In [ ]:
TOY = {
    "d1": "students learn english grammar with online exercises",
    "d2": "online courses help teachers plan english lessons",
    "d3": "search engines rank web pages by relevance",
    "d4": "teachers use search engines to find lesson materials",
    "d5": "grammar exercises improve writing in english",
    "d6": "an inverted index maps each word to documents",
    "d7": "distributed search splits the index into shards",
    "d8": "shards are stored on different servers with replicas",
}
TOY_SHARDS = {"шард A": ["d1", "d2", "d5", "d6"], "шард B": ["d3", "d4", "d7", "d8"]}
tok = lambda s: re.findall(r"[a-z]+", s.lower())

def build_index(doc_ids):
    index = defaultdict(dict)                  # терм → {документ: частота}
    for doc_id in doc_ids:
        for term in tok(TOY[doc_id]):
            ... # TODO 1 ← впишите: index[term][doc_id] = index[term].get(doc_id, 0) + 1
    return index

def idf(df, n_docs):
    value = ...    # TODO 2 ← впишите: math.log(1 + (n_docs - df + 0.5) / (df + 0.5))
    return value

def bm25_toy(query, index, n_docs, lengths, k1=1.2, b=0.75):
    avgdl = sum(lengths.values()) / len(lengths); scores = defaultdict(float)
    for t in tok(query):
        postings = index.get(t, {})
        for d, tf in postings.items():
            scores[d] += idf(len(postings), n_docs) * tf * (k1 + 1) / (tf + k1 * (1 - b + b * lengths[d] / avgdl))
    return dict(sorted(scores.items(), key=lambda x: -x[1]))

global_index = build_index(TOY)
check_filled("шаг 2, TODO 1", {"индекс": ... if not global_index else 1})
check_filled("шаг 2, TODO 2", {"idf": idf(1, 8)})
lengths = {d: len(tok(t)) for d, t in TOY.items()}
print("Списки вхождений (фрагмент):")
for t in ["english", "search", "shards", "grammar"]:
    print(f"  {t:8s} → {global_index[t]}")

query = "english grammar search"
print(f"\nЗапрос: «{query}»")
rows = []
for t in tok(query):
    row = {"терм": t, "df глоб.": len(global_index.get(t, {})), "IDF глоб.": round(idf(len(global_index.get(t, {})), 8), 3)}
    for name, ids in TOY_SHARDS.items():
        loc = build_index(ids); df_loc = len(loc.get(t, {}))
        row[f"df {name}"] = df_loc; row[f"IDF {name}"] = round(idf(df_loc, len(ids)), 3) if df_loc else "—"
    rows.append(row)
display(pd.DataFrame(rows))

res_global = bm25_toy(query, global_index, 8, lengths)
res_local = {}
for name, ids in TOY_SHARDS.items():
    res_local.update(bm25_toy(query, build_index(ids), len(ids), {d: lengths[d] for d in ids}))
cmp = pd.DataFrame({"BM25 (глобальная статистика)": res_global, "BM25 (локальная статистика шарда)": res_local}).round(3)
cmp["шард"] = [next(n for n, ids in TOY_SHARDS.items() if d in ids) for d in cmp.index]
display(cmp.sort_values("BM25 (глобальная статистика)", ascending=False))

**Как читать результат.** Слово `english` встречается в трёх документах коллекции, но все три — в шарде A; в шарде B его нет совсем. Поэтому IDF этого слова **в шарде A меньше**, чем в коллекции: внутри шарда оно кажется частым. А слово `search` в шарде B встречается в трёх документах из четырёх, и его локальный IDF тоже ниже глобального. В итоге документы получают разные оценки в зависимости от того, где лежат, — и сравнение «сырых» оценок разных шардов становится нечестным.

**Что получено.** Допишите:

> Слово `english` встречается в ___ документах коллекции; глобальный IDF = ___, IDF в шарде A = ___. Лучший документ по глобальной статистике — ___ (оценка ___), по локальной — ___ (оценка ___). Порядок документов ___ (изменился / не изменился), потому что ___.

---
# Шаг 3. Индекс всей коллекции и централизованный эталон

**Ячейка готова — просто запустите.** Строим индекс 3 633 документов в виде разреженной матрицы «документ × терм» (это тот же инвертированный индекс, только записанный компактно), считаем BM25 для каждого тестового запроса и измеряем качество **одного большого индекса** — с ним дальше будем сравнивать распределённые варианты.

In [ ]:
TOKEN = r"(?u)\b[a-zA-Z][a-zA-Z]+\b"
vectorizer = CountVectorizer(token_pattern=TOKEN, lowercase=True, stop_words="english")
TF = vectorizer.fit_transform(docs["content"]).tocsc().astype(np.float32)   # документы × термы
VOCAB = vectorizer.vocabulary_; analyze = vectorizer.build_analyzer()
DOC_IDS = docs["doc_id"].values; N_DOCS = TF.shape[0]
DL = np.asarray(TF.sum(axis=1)).ravel()                    # длины документов
DF_GLOBAL = np.asarray((TF > 0).sum(axis=0)).ravel()       # глобальные df

def bm25(query_terms, tf, dl, df, n_docs, avgdl, k1=1.2, b=0.75):
    """BM25 для всех документов матрицы tf (строки — документы данного индекса или шарда)."""
    scores = np.zeros(tf.shape[0], dtype=np.float32)
    for t in query_terms:
        j = VOCAB.get(t)
        if j is None or df[j] == 0:
            continue
        idf_t = math.log(1 + (n_docs - df[j] + 0.5) / (df[j] + 0.5))
        col = tf[:, j].toarray().ravel()
        scores += idf_t * col * (k1 + 1) / (col + k1 * (1 - b + b * dl / avgdl))
    return scores

def ndcg_at(ranked, qid, k):
    rel = REL[qid]
    dcg = sum((2 ** rel.get(d, 0) - 1) / math.log2(i + 2) for i, d in enumerate(ranked[:k]))
    ideal = sorted((2 ** r - 1 for r in rel.values()), reverse=True)[:k]
    idcg = sum(g / math.log2(i + 2) for i, g in enumerate(ideal))
    return dcg / idcg if idcg else 0.0

def recall_at(ranked, qid, k=100):
    return len(set(ranked[:k]) & set(REL[qid])) / len(REL[qid])

t0 = time.perf_counter()
CENTRAL = {}
for q in test_q:
    s = bm25(analyze(queries[q]), TF, DL, DF_GLOBAL, N_DOCS, DL.mean())
    CENTRAL[q] = DOC_IDS[np.argsort(-s)[:100]]
t_central = time.perf_counter() - t0
nd_central = np.array([ndcg_at(CENTRAL[q], q, K) for q in test_q])
print(f"Индекс: {N_DOCS} документов × {TF.shape[1]} термов; средняя длина {DL.mean():.0f} слов")
print(f"Централизованный BM25: NDCG@{K} = {nd_central.mean():.3f}, Recall@100 = {np.mean([recall_at(CENTRAL[q], q) for q in test_q]):.3f}; "
      f"запросов с NDCG = 0: {int((nd_central == 0).sum())}; время {t_central:.1f} с")

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(nd_central, bins=20, color=C_MAIN)
ax.axvline(nd_central.mean(), color=C_ALT, ls="--", label=f"среднее {nd_central.mean():.3f}")
ax.set(title=f"Распределение NDCG@{K} по запросам: единый индекс", xlabel=f"NDCG@{K}", ylabel="запросов"); ax.legend()
plt.tight_layout(); plt.show()

Q = CFG["query_id"]
print(f"Запрос варианта «{queries[Q]}»: NDCG@{K} = {ndcg_at(CENTRAL[Q], Q, K):.3f}")
display(pd.DataFrame({"место": range(1, 6), "документ": CENTRAL[Q][:5],
                      "заголовок": docs.set_index("doc_id").loc[CENTRAL[Q][:5], "title"].values,
                      "релевантность": [REL[Q].get(d, 0) for d in CENTRAL[Q][:5]]}))

**Как читать результат.** NDCG около 0,3 — нормальный уровень для BM25 на этой коллекции: запросы короткие, а релевантные документы часто используют другие слова (синонимы, научные термины). Столбик у нуля на гистограмме — запросы, по которым BM25 не нашёл ничего релевантного в топ-k; распределение их не исправит. Это эталон: распределённый поиск в лучшем случае может **сравняться** с ним.

**Что получено.** Допишите:

> Централизованный BM25: NDCG@___ = ___, Recall@100 = ___; запросов с NDCG = 0 — ___. По запросу моего варианта NDCG = ___; из первых пяти документов релевантны ___.

---
# Шаг 4. Разбиение коллекции на шарды (TODO 3)

Функция `make_shards` раскладывает документы по шардам одним из трёх способов:

* **random** — случайно (так делают Elasticsearch и большинство систем: шард выбирается по хешу идентификатора);
* **topic** — по темам: алгоритм K-Means группирует документы с похожими словами (так устроен selective search);
* **length** — по длине документа (короткие отдельно, длинные отдельно) — пример неудачного «административного» разбиения.

**Задание (TODO 3).** Допишите строку `shard_of = make_shards(N_SHARDS, STRATEGY)`.

In [ ]:
from sklearn.cluster import KMeans

def make_shards(n, how, seed=42):
    if how == "random":
        return np.random.default_rng(seed).integers(0, n, N_DOCS)
    if how == "length":
        return pd.qcut(DL, n, labels=False, duplicates="drop").astype(int)
    if how == "topic":
        X = TfidfVectorizer(token_pattern=TOKEN, stop_words="english", max_features=20000, sublinear_tf=True).fit_transform(docs["content"])
        return KMeans(n_clusters=n, random_state=seed, n_init=5).fit_predict(X)
    raise ValueError(how)

shard_of = ...   # TODO 3 ← впишите: make_shards(N_SHARDS, STRATEGY)
check_filled("шаг 4, TODO 3", {"shard_of": shard_of})

SHARDS = []
for s in range(N_SHARDS):
    idx = np.where(shard_of == s)[0]; tf_s = TF[idx]
    SHARDS.append(dict(idx=idx, tf=tf_s, dl=DL[idx], df=np.asarray((tf_s > 0).sum(axis=0)).ravel(),
                       n=len(idx), avgdl=DL[idx].mean(), ttf=np.asarray(tf_s.sum(axis=0)).ravel()))
sizes = [S["n"] for S in SHARDS]
terms = np.array(vectorizer.get_feature_names_out())
info = pd.DataFrame({"документов": sizes, "средняя длина": [round(S["avgdl"]) for S in SHARDS],
                     "характерные слова": [", ".join(terms[np.argsort(-(S["df"] / S["n"] - DF_GLOBAL / N_DOCS))[:6]]) for S in SHARDS]},
                    index=[f"шард {s}" for s in range(N_SHARDS)])
display(info)

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
ax[0].bar(info.index, sizes, color=C_MAIN); ax[0].set(title=f"Размеры шардов ({STRAT_RU[STRATEGY]})", ylabel="документов")
ax[0].tick_params(axis="x", rotation=45)
# локальный и глобальный IDF самых частых слов
top_terms = np.argsort(-DF_GLOBAL)[:300]
glob_idf = np.log(1 + (N_DOCS - DF_GLOBAL[top_terms] + 0.5) / (DF_GLOBAL[top_terms] + 0.5))
for s, S in enumerate(SHARDS[:4]):
    loc_idf = np.log(1 + (S["n"] - S["df"][top_terms] + 0.5) / (S["df"][top_terms] + 0.5))
    ax[1].scatter(glob_idf, loc_idf, s=8, alpha=0.5, label=f"шард {s}")
lim = [0, max(glob_idf.max(), 6)]; ax[1].plot(lim, lim, color="black", lw=1, ls="--")
ax[1].set(title="IDF 300 частых слов: глобальный vs локальный", xlabel="глобальный IDF", ylabel="IDF в шарде"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
spread = np.std([np.log(1 + (S["n"] - S["df"][top_terms] + 0.5) / (S["df"][top_terms] + 0.5)) for S in SHARDS], axis=0).mean()
print(f"Средний разброс локального IDF между шардами: {spread:.3f}")

**Как читать результат.** Таблица показывает, чем шарды отличаются: при тематическом разбиении у каждого шарда свои «характерные слова» (например, о раке, о сердце, о детях), при случайном — характерные слова почти случайны, при разбиении по длине различаются средние длины. На правом графике точки на пунктирной диагонали — слова, у которых IDF в шарде совпадает с глобальным; чем дальше точки от диагонали, тем сильнее локальная статистика искажает BM25. При случайном разбиении точки лежат почти на диагонали, при тематическом — разлетаются.

**Что получено.** Допишите:

> Шарды моего варианта (___ разбиение): размеры от ___ до ___ документов. Характерные слова самого крупного шарда: ___. Точки на графике IDF ___ (близко к диагонали / разбросаны); средний разброс локального IDF = ___. Я ожидаю, что слияние «сырых» локальных оценок ___.

---
# Шаг 5. Брокер и слияние результатов (TODO 4)

Брокер рассылает запрос всем шардам; каждый шард считает BM25 **по своей локальной статистике** и возвращает топ-`K_LOCAL`. Для сравнения шард также сообщает оценку по **глобальной** статистике (как в режиме `dfs_query_then_fetch` Elasticsearch). Затем брокер сливает списки шестью способами.

**Задание (TODO 4).** Допишите функцию слияния **RRF** — она учитывает только место документа в списке своего шарда:
`return 1 / (60 + rank)`

In [ ]:
def shard_search(query_terms, S, k_local):
    """Поиск в одном шарде: топ-k по локальной статистике + оценка по глобальной."""
    local = bm25(query_terms, S["tf"], S["dl"], S["df"], S["n"], S["avgdl"])
    glob = bm25(query_terms, S["tf"], S["dl"], DF_GLOBAL, N_DOCS, DL.mean())
    top = np.argsort(-local)[:k_local]
    return pd.DataFrame({"doc": DOC_IDS[S["idx"][top]], "local": local[top], "glob": glob[top], "rank": np.arange(1, len(top) + 1)})

def merge(L, method):
    g = L.groupby("shard")["local"]
    if method == "сырые (локальная стат.)":
        score = L["local"]
    elif method == "сырые (глобальная стат.)":
        score = L["glob"]
    elif method == "min-max":
        score = (L["local"] - g.transform("min")) / (g.transform("max") - g.transform("min") + 1e-9)
    elif method == "z-оценка":
        score = (L["local"] - g.transform("mean")) / (g.transform("std").fillna(0) + 1e-9)
    elif method == "RRF":
        score = rrf(L["rank"])
    elif method == "круговой":
        score = -L["rank"] - L["shard"] / 100          # первые места всех шардов, затем вторые…
    return L.assign(score=np.asarray(score)).sort_values(["score", "glob"], ascending=False)["doc"].values

def rrf(rank):
    """Reciprocal Rank Fusion: оценка зависит только от места документа в списке своего шарда."""
    return ...   # TODO 4 ← впишите: 1 / (60 + rank)

check_filled("шаг 5, TODO 4", {"rrf": rrf(1)})

METHODS = ["сырые (локальная стат.)", "сырые (глобальная стат.)", "min-max", "z-оценка", "RRF", "круговой"]
def broker(qid, shards_to_ask=None, k_local=K_LOCAL):
    """Брокер: рассылает запрос шардам и собирает их списки (scatter–gather)."""
    terms_q = analyze(queries[qid]); ask = range(len(SHARDS)) if shards_to_ask is None else shards_to_ask
    return pd.concat([shard_search(terms_q, SHARDS[s], k_local).assign(shard=s) for s in ask], ignore_index=True)

t0 = time.perf_counter()
results = {m: [] for m in METHODS}
for q in test_q:
    L = broker(q)
    for m in METHODS:
        results[m].append(ndcg_at(merge(L, m), q, K))
t_dist = time.perf_counter() - t0
merge_table = pd.DataFrame({"NDCG@%d" % K: {m: np.mean(v) for m, v in results.items()},
                            "запросов хуже эталона": {m: int((np.array(v) < nd_central - 1e-9).sum()) for m, v in results.items()}}).round(3)
merge_table.loc["единый индекс (эталон)"] = [round(nd_central.mean(), 3), 0]
display(merge_table)

fig, ax = plt.subplots(figsize=(9, 3.6))
vals = merge_table.iloc[:, 0]
ax.barh(vals.index, vals.values, color=[C_ALT if i == len(vals) - 1 else C_MAIN for i in range(len(vals))])
for i, v in enumerate(vals.values): ax.text(v + 0.003, i, f"{v:.3f}", va="center", fontsize=9)
ax.set(title=f"Слияние результатов: {N_SHARDS} шардов, {STRAT_RU[STRATEGY]} разбиение, топ-{K_LOCAL} от шарда", xlabel=f"NDCG@{K}")
ax.invert_yaxis(); plt.tight_layout(); plt.show()
print(f"Время распределённого поиска (все запросы, все методы): {t_dist:.1f} с")

**Как читать результат.**

* **Сырые оценки с глобальной статистикой** почти всегда совпадают с эталоном: если шарды «договорились» об общих $N$, $df$ и $avgdl$, распределение по серверам **не портит** качество (при условии, что каждый шард возвращает не меньше k документов).
* **Сырые локальные оценки** немного хуже — сказывается искажение IDF из шага 4; при тематическом разбиении потери больше.
* **min-max, z-оценка, RRF, круговой** игнорируют, что шарды неравноценны: лучший документ «пустого» по теме шарда получает высокую итоговую оценку. Поэтому при тематическом разбиении они проигрывают, а при случайном — почти не отличаются. Эти методы нужны, когда оценки шардов действительно несравнимы (разные поисковые системы в метапоиске).

**Что получено.** Допишите:

> Лучший метод слияния — ___ (NDCG@___ = ___), эталон — ___. Сырые локальные оценки дают ___, это на ___ меньше эталона. Худший метод — ___ (___), потому что ___. Моя гипотеза ___ (подтвердилась / не подтвердилась).

---
# Шаг 6. Выбор шардов: качество, стоимость и задержка (TODO 5)

**Ячейка готова до строки TODO.** Брокер оценивает каждый шард как один «большой документ» (сумма всех его текстов) по BM25 и опрашивает только $s$ лучших шардов. Для каждого $s$ считаем NDCG, долю оценённых документов и **задержку**: время ответа шарда моделируется как пропорциональное его размеру плюс случайная задержка сети; брокер ждёт **самый медленный** из опрошенных шардов.

**Задание (TODO 5).** Посмотрите на график и **сами выберите** число опрашиваемых шардов: впишите `S_SELECT = ...` (в таблице вариантов указано рекомендуемое значение `CFG["s_select"]`, можно выбрать другое — обоснуйте в выводе).

In [ ]:
def shard_scores(query_terms):
    """Оценка шардов как «больших документов»: чем чаще в шарде встречаются слова запроса, тем выше."""
    lens = np.array([S["ttf"].sum() for S in SHARDS]); avg = lens.mean(); out = []
    for S, length in zip(SHARDS, lens):
        s = 0.0
        for t in query_terms:
            j = VOCAB.get(t)
            if j is None or S["df"][j] == 0:
                continue
            idf_t = math.log(1 + (N_DOCS - DF_GLOBAL[j] + 0.5) / (DF_GLOBAL[j] + 0.5)); x = S["df"][j]
            s += idf_t * x * 2.2 / (x + 1.2 * (0.25 + 0.75 * length / avg))
        out.append(s)
    return np.array(out)

rng = np.random.default_rng(V)
def latency_ms(shard_ids):
    """Модель задержки: 2 мс + 0,01 мс на документ шарда + случайная сетевая задержка; брокер ждёт самого медленного."""
    return max(2 + 0.01 * SHARDS[s]["n"] + rng.exponential(3) for s in shard_ids) + 1.0

curve = []
for s_sel in range(1, N_SHARDS + 1):
    nd, cost, lat = [], [], []
    for q in test_q:
        order = np.argsort(-shard_scores(analyze(queries[q])))[:s_sel]
        L = broker(q, shards_to_ask=list(order))
        nd.append(ndcg_at(merge(L, "сырые (глобальная стат.)"), q, K))
        cost.append(sum(SHARDS[s]["n"] for s in order) / N_DOCS); lat.append(latency_ms(order))
    curve.append(dict(s=s_sel, ndcg=np.mean(nd), cost=np.mean(cost), lat_p50=np.median(lat), lat_p95=np.percentile(lat, 95)))
curve = pd.DataFrame(curve).round(3); display(curve)

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
ax[0].plot(curve["cost"] * 100, curve["ndcg"], "o-", color=C_MAIN)
for _, r in curve.iterrows(): ax[0].annotate(f"s={int(r.s)}", (r.cost * 100, r.ndcg), textcoords="offset points", xytext=(4, -10), fontsize=8)
ax[0].axhline(nd_central.mean(), color=C_ALT, ls="--", label="единый индекс")
ax[0].set(title="Качество и стоимость при выборе шардов", xlabel="оценено документов, %", ylabel=f"NDCG@{K}"); ax[0].legend()
ax[1].plot(curve["s"], curve["lat_p50"], "o-", color=C_MAIN, label="медиана (p50)")
ax[1].plot(curve["s"], curve["lat_p95"], "s--", color=C_BAD, label="95-й перцентиль (p95)")
ax[1].set(title="Модельная задержка брокера", xlabel="опрошено шардов s", ylabel="мс"); ax[1].legend()
plt.tight_layout(); plt.show()

S_SELECT = ...   # TODO 5 ← впишите выбранное число шардов, например: CFG["s_select"]
check_filled("шаг 6, TODO 5", {"S_SELECT": S_SELECT})
S_SELECT = int(S_SELECT); assert 1 <= S_SELECT <= N_SHARDS, f"TODO 5: s должно быть от 1 до {N_SHARDS}"
row = curve.set_index("s").loc[S_SELECT]
print(f"Выбрано s = {S_SELECT}: NDCG@{K} = {row.ndcg:.3f} ({row.ndcg / nd_central.mean():.0%} эталона) при {row.cost:.0%} оценённых документов; p95 задержки {row.lat_p95:.1f} мс")

**Как читать результат.** Левый график — компромисс: каждая точка — число опрашиваемых шардов. Чем левее и выше точка, тем выгоднее: высокое качество при малой работе. При **тематическом** разбиении первые 2–3 шарда дают почти всё качество, потому что релевантные документы сосредоточены в «своей» теме. При **случайном** разбиении релевантные документы равномерно распределены, и выбор шардов просто теряет их пропорционально. Правый график — задержка: брокер ждёт самого медленного шарда, поэтому каждый дополнительный шард повышает «хвост» (p95).

**Что получено.** Допишите:

> При опросе всех ___ шардов NDCG@___ = ___. Я выбрал(а) s = ___: качество ___ (___ % эталона) при ___ % оценённых документов; p95 задержки снизился с ___ до ___ мс. Выбор шардов ___ (выгоден / невыгоден) для моего разбиения, потому что ___.

---
# Шаг 7. Отказ шарда и репликация

**Ячейка готова — просто запустите.** Сервер шарда № `CFG["failed"]` выходит из строя. Сравниваем три ситуации: все шарды работают; шард недоступен, реплики нет; шард недоступен, но брокер обращается к **реплике** — копии шарда на другом сервере.

In [ ]:
F = CFG["failed"] % N_SHARDS
alive = [s for s in range(N_SHARDS) if s != F]
nd_fail = np.array([ndcg_at(merge(broker(q, shards_to_ask=alive), "сырые (глобальная стат.)"), q, K) for q in test_q])
nd_full = np.array(results["сырые (глобальная стат.)"])
lost = int(((nd_full > 0) & (nd_fail == 0)).sum())
fail_table = pd.DataFrame({f"NDCG@{K}": [nd_full.mean(), nd_fail.mean(), nd_full.mean()],
                           "запросов с NDCG = 0": [int((nd_full == 0).sum()), int((nd_fail == 0).sum()), int((nd_full == 0).sum())],
                           "серверов": [N_SHARDS, N_SHARDS, 2 * N_SHARDS]},
                          index=["все шарды работают", f"отказ шарда {F}, без реплики", f"отказ шарда {F}, есть реплика"]).round(3)
display(fail_table)
print(f"Шард {F} содержит {SHARDS[F]['n']} документов ({SHARDS[F]['n'] / N_DOCS:.0%} коллекции); "
      f"запросов, потерявших все релевантные документы в топ-{K}: {lost}")
fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(fail_table.index, fail_table.iloc[:, 0], color=[C_MAIN, C_BAD, C_ALT])
for i, v in enumerate(fail_table.iloc[:, 0]): ax.text(v + 0.003, i, f"{v:.3f}", va="center", fontsize=9)
ax.set(title="Отказ шарда и репликация", xlabel=f"NDCG@{K}"); ax.invert_yaxis(); plt.tight_layout(); plt.show()

**Как читать результат.** При случайном разбиении отказ одного шарда снижает качество понемногу для всех запросов. При тематическом — запросы «его» темы могут потерять почти все релевантные документы, и число запросов с нулевым NDCG растёт. Реплика возвращает качество полностью, но требует вдвое больше серверов — так устроены `number_of_replicas` в Elasticsearch и OpenSearch.

**Что получено.** Допишите:

> Отказ шарда ___ (___ % коллекции) снизил NDCG@___ с ___ до ___; ___ запросов потеряли все релевантные документы в топ-___. С репликой качество ___, но серверов нужно ___. Для моего разбиения отказ ___ (опасен / не очень опасен), потому что ___.

---
# Шаг 8. Анализ выдачи по запросу варианта (английский язык)

**Ячейка готова — просто запустите.** Показываем топ-10 итоговой выдачи брокера по вашему запросу: из какого шарда пришёл документ, его заголовок и оценку асессора. Прочитайте заголовки — это англоязычные названия научных статей.

In [ ]:
L = broker(Q)
ranked = merge(L, "сырые (глобальная стат.)")[:10]
meta = docs.set_index("doc_id")
top10 = pd.DataFrame({"место": range(1, 11), "шард": [int(L.loc[L["doc"] == d, "shard"].iat[0]) for d in ranked],
                      "заголовок": meta.loc[ranked, "title"].values, "оценка асессора": [REL[Q].get(d, 0) for d in ranked]})
print(f"Запрос: «{queries[Q]}» · NDCG@{K} = {ndcg_at(list(ranked), Q, K):.3f} · релевантных в коллекции: {len(REL[Q])}")
print("Слова запроса после обработки:", analyze(queries[Q]))
display(top10)
missed = [d for d, r in sorted(REL[Q].items(), key=lambda x: -x[1]) if d not in set(merge(L, "сырые (глобальная стат.)")[:K])][:3]
print("\nПримеры релевантных документов, которые не попали в топ:")
for d in missed:
    print(f"  [{REL[Q][d]}] {meta.loc[d, 'title']}  (шард {int(shard_of[np.where(DOC_IDS == d)[0][0]])})")

**Как читать результат.** Обратите внимание на **слова запроса после обработки**: служебные слова удалены, а слова не приведены к начальной форме (`arteries` и `artery` — разные термы). Сравните заголовки найденных и пропущенных релевантных документов: часто пропущенные используют синонимы или научные термины вместо слов запроса — это ограничение лексического поиска, которое распределение не исправит.

**Что получено.** Допишите (заголовки можно цитировать на английском):

> По запросу «___» в топ-10 ___ релевантных документов, они пришли из шардов ___. Пример релевантного документа: «___» (перевод: «___»). Пример пропущенного: «___» — не найден, потому что ___. Чтобы улучшить поиск, можно ___ (например, стемминг, синонимы).

---
# Итог обязательной части. Самопроверка и вывод

**Ячейка готова — просто запустите.**

In [ ]:
checks = {
    "Заполнены ФИО и группа":                     STUDENT_NAME != "Иванов Иван Иванович",
    "Коллекция загружена (3633 документа)":       len(docs) == 3633,
    "TODO 1–2: индекс и IDF на игрушечном примере": len(global_index) > 0 and abs(idf(1, 8) - math.log(1 + 7.5 / 1.5)) < 1e-9,
    "TODO 3: коллекция разбита на шарды":          len(SHARDS) == N_SHARDS and sum(sizes) == N_DOCS,
    "TODO 4: слияние RRF":                         rrf(1) is not ... and abs(rrf(1) - 1 / 61) < 1e-12,
    "TODO 5: выбрано число шардов":                isinstance(S_SELECT, int),
    "Проанализированы отказ и выдача":             "fail_table" in globals() and "top10" in globals(),
}
for k_, ok in checks.items():
    print(("OK " if ok else "НЕТ"), k_)
print(f"\nОбязательная часть: выполнено {sum(checks.values())} из {len(checks)}")
summary = pd.DataFrame({
    "показатель": [f"NDCG@{K} единого индекса", "лучший метод слияния", "NDCG лучшего метода", "сырые локальные оценки",
                   f"выбор s = {S_SELECT} шардов", "отказ шарда без реплики"],
    "значение": [f"{nd_central.mean():.3f}", merge_table.iloc[:-1, 0].idxmax(), f"{merge_table.iloc[:-1, 0].max():.3f}",
                 f"{merge_table.loc['сырые (локальная стат.)'].iloc[0]:.3f}", f"{row.ndcg:.3f} при {row.cost:.0%} документов",
                 f"{nd_fail.mean():.3f}"]})
display(summary)

### ✍️ Итоговый вывод (6–8 предложений)

> 1) Как локальная статистика искажает BM25 (шаг 2 и 4)? 2) Сколько потерял распределённый поиск по сравнению с единым индексом и при каком способе слияния? 3) Почему методы нормализации проиграли (или нет) в вашем разбиении? 4) Какое число шардов вы выбрали и чем заплатили? 5) Чем опасен отказ шарда и как помогает реплика? 6) Что вы заметили в англоязычной выдаче? 7) Как это связано с режимами `query_then_fetch` и `dfs_query_then_fetch` в Elasticsearch?
>
> _[ваш вывод]_

---
# Усложнение (+4 балла). Модель ранжирования на брокере (по материалам семинара)

В **федеративном поиске** брокер опрашивает разные поисковые системы, у каждой своя формула ранжирования, и их оценки несравнимы. Решение из семинара — **обучение ранжированию** (Learning to Rank): модель учится по оценкам асессоров, как комбинировать много признаков документа, и переупорядочивает кандидатов, собранных брокером.

**Без машинного обучения на пальцах.** Модель — это набор простых правил «если признак больше порога, прибавить/убавить оценку» (деревья решений). Таких деревьев сотни; каждое следующее исправляет ошибки предыдущих (**градиентный бустинг**, библиотека CatBoost). Модель учится на запросах **обучающей** выборки, а момент остановки выбирается по **валидационной** — отдельным запросам, которые модель не видела; **тестовая** выборка используется только для итоговой оценки (в семинаре ради простоты остановка шла по тесту — так делать нельзя).

| Подход | Что сравнивает модель | Функции потерь CatBoost |
|---|---|---|
| поточечный (pointwise) | предсказывает оценку каждого документа отдельно | `RMSE`, `QueryRMSE` |
| попарный (pairwise) | учится правильно упорядочивать пары документов | `PairLogit` |
| списочный (listwise) | оптимизирует качество всего списка (близко к NDCG) | `YetiRank`, `LambdaMart` |

**Данные — MSLR из семинара:** 175 запросов, по ~100 документов на запрос, 136 признаков, оценки 0–4. Признаки имеют смысл: BM25, TF-IDF, языковые модели (LMIR) по разным полям, PageRank, длина URL, клики.

**Сценарий.** Документы каждого запроса разложены по `N_SHARDS` «поисковым системам»; каждая ранжирует **своей** формулой (BM25, TF-IDF или LMIR), возвращает топ-`K_LOCAL`, брокер сливает их min-max нормализацией. Затем модель переранжирует кандидатов.

In [ ]:
try:
    import catboost
except ImportError:                                     # в Colab CatBoost не установлен
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "catboost"], check=True)
    import catboost
from catboost import CatBoost, Pool, datasets, utils

mslr_train, mslr_test = datasets.msrank_10k()
FEATS = [f"f{i}" for i in range(1, 137)]
mslr_train.columns = mslr_test.columns = ["label", "qid"] + FEATS
FIELDS = ["body", "anchor", "title", "url", "whole"]
GROUPS = ["covered query terms", "covered query term ratio", "stream length", "IDF", "sum TF", "min TF", "max TF", "mean TF",
          "variance TF", "sum normalized TF", "min normalized TF", "max normalized TF", "mean normalized TF", "variance normalized TF",
          "sum TF-IDF", "min TF-IDF", "max TF-IDF", "mean TF-IDF", "variance TF-IDF", "boolean model", "vector space model",
          "BM25", "LMIR.ABS", "LMIR.DIR", "LMIR.JM"]          # признаки 1–125: группа × поле (официальный список MSLR)
FEATURE_NAMES = {f"f{5 * g + i + 1}": f"{gr} ({fl})" for g, gr in enumerate(GROUPS) for i, fl in enumerate(FIELDS)}
FEATURE_NAMES.update({"f126": "URL slash count", "f127": "URL length", "f128": "inlinks", "f129": "outlinks", "f130": "PageRank",
                      "f131": "SiteRank", "f132": "QualityScore", "f133": "QualityScore2", "f134": "query-url clicks", "f135": "url clicks", "f136": "url dwell time"})
name = lambda f: FEATURE_NAMES.get(f, f)

# обучающая / валидационная выборки: делим запросы train, тест не трогаем
rng_b = np.random.default_rng(V)
val_q = set(rng_b.choice(mslr_train["qid"].unique(), 20, replace=False))
trn, val = mslr_train[~mslr_train["qid"].isin(val_q)], mslr_train[mslr_train["qid"].isin(val_q)]
print(f"MSLR: train {trn['qid'].nunique()} запросов, valid {val['qid'].nunique()}, test {mslr_test['qid'].nunique()}; оценки 0–4")

def ndcg_mslr(df, score, k=K):
    return utils.eval_metric(df["label"].values, np.asarray(score), f"NDCG:top={k};type=Exp", group_id=df["qid"].values)[0]

SCORERS = ["f110", "f75", "f120"]   # BM25, TF-IDF, LMIR по всему документу — у каждой «поисковой системы» своя формула
def federate(df, n=N_SHARDS, k_local=K_LOCAL, seed=V):
    d = df.copy(); d["engine"] = np.random.default_rng(seed).integers(0, n, len(d))
    d["local"] = [d.at[i, SCORERS[e % len(SCORERS)]] for i, e in zip(d.index, d["engine"])]
    d["r"] = d.groupby(["qid", "engine"])["local"].rank(ascending=False, method="first")
    cand = d[d["r"] <= k_local].copy(); g = cand.groupby(["qid", "engine"])["local"]
    cand["merged"] = (cand["local"] - g.transform("min")) / (g.transform("max") - g.transform("min")).replace(0, 1)
    d["merged"] = -1e9; d.loc[cand.index, "merged"] = cand["merged"]
    return d, cand

fed_test, cand_test = federate(mslr_test)
print(f"BM25 по единому индексу: NDCG@{K} = {ndcg_mslr(mslr_test, mslr_test['f110']):.3f}")
print(f"Федеративное слияние (min-max): NDCG@{K} = {ndcg_mslr(fed_test, fed_test['merged']):.3f}; кандидатов на запрос: {len(cand_test) / mslr_test['qid'].nunique():.0f}")

### Б1–Б2. Обучение модели (TODO Б1–Б2)

**Задание.**

* **TODO Б1** — создайте валидационный набор (по нему выбирается момент остановки): `Pool(val[FEATS], val["label"], group_id=val["qid"])`
* **TODO Б2** — укажите функцию потерь своего варианта: `CFG["ltr_loss"]`

In [ ]:
pool_train = Pool(trn[FEATS], trn["label"], group_id=trn["qid"])
pool_valid = ...   # TODO Б1 ← впишите: Pool(val[FEATS], val["label"], group_id=val["qid"])
LOSS = ...         # TODO Б2 ← впишите: CFG["ltr_loss"]
check_filled("усложнение, TODO Б1–Б2", {"pool_valid": pool_valid, "LOSS": LOSS})

model = CatBoost({"loss_function": LOSS, "iterations": 1000, "early_stopping_rounds": 100,
                  "eval_metric": f"NDCG:top={K};type=Exp", "random_seed": V, "verbose": 0, "train_dir": "/tmp/catboost_info"})
t0 = time.perf_counter()
model.fit(pool_train, eval_set=pool_valid, use_best_model=True)
print(f"Модель {LOSS}: деревьев {model.tree_count_}, обучение {time.perf_counter() - t0:.1f} с, "
      f"NDCG@{K} на валидации {ndcg_mslr(val, model.predict(val[FEATS])):.3f}")

fed_test["ltr"] = -1e9
fed_test.loc[cand_test.index, "ltr"] = model.predict(cand_test[FEATS])
ltr_table = pd.Series({
    "BM25, единый индекс": ndcg_mslr(mslr_test, mslr_test["f110"]),
    "федеративное слияние (min-max)": ndcg_mslr(fed_test, fed_test["merged"]),
    f"брокер + модель {LOSS}": ndcg_mslr(fed_test, fed_test["ltr"]),
    f"модель {LOSS}, единый индекс": ndcg_mslr(mslr_test, model.predict(mslr_test[FEATS])),
}).round(3)
display(ltr_table.to_frame(f"NDCG@{K} (test)"))

imp = pd.Series(model.get_feature_importance(pool_train, type="PredictionValuesChange"), index=[name(f) for f in FEATS]).sort_values(ascending=False)
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
ax[0].barh(ltr_table.index, ltr_table.values, color=[C_ALT, C_BAD, C_MAIN, "#9b8ac4"])
for i, v in enumerate(ltr_table.values): ax[0].text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=9)
ax[0].set(title="Брокер с моделью ранжирования (MSLR, test)", xlabel=f"NDCG@{K}"); ax[0].invert_yaxis()
imp.head(10)[::-1].plot.barh(ax=ax[1], color=C_MAIN); ax[1].set(title="10 самых важных признаков модели", xlabel="важность, %")
plt.tight_layout(); plt.show()

**Как читать результат.** Федеративное слияние сильно уступает единому индексу: формулы разных «поисковых систем» несравнимы даже после нормализации. Модель на брокере использует **все** признаки кандидата и возвращает качество выше исходного BM25 — почти до уровня модели на едином индексе. Важность признаков показывает, на что опирается модель: обычно это BM25 и языковые модели по разным полям, клики и качество страницы. Важность — это вклад в решения модели, а не причина релевантности.

**Что получено (усложнение).** Допишите:

> Функция потерь ___ (подход ___). Федеративное слияние: NDCG@___ = ___; брокер с моделью: ___; BM25 единого индекса: ___; модель на едином индексе: ___. Модель вернула ___ % разрыва между федеративным слиянием и моделью на едином индексе. Самые важные признаки: ___, ___, ___. Валидационная выборка нужна, потому что ___.

In [ ]:
print("Усложнение:", "выполнено" if "ltr_table" in globals() else "не выполнялось")
if "ltr_table" in globals():
    gap = (ltr_table.iloc[2] - ltr_table.iloc[1]) / max(ltr_table.iloc[3] - ltr_table.iloc[1], 1e-9)
    print(f"  функция потерь {LOSS}: брокер + модель = {ltr_table.iloc[2]:.3f}; возвращено {gap:.0%} разрыва")